# Lab 7.5 &mdash; Route: Decide What Happens Next

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Map an intent to a team from a fixed set (with an escape hatch)
- Escalate negative or unknown cases to a human
- Emit a label that drives the next branch of the workflow

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Route — decide what happens next](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-05"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
**Route** decides what happens next and emits a **label from a fixed set** that drives a branch
(deck slide 11): which team, how urgent, auto-handle or escalate. The **closed list** is the trick
that makes an LLM a reliable classifier &mdash; include an **escape hatch** (`other` / `unsure`), and
a **fallback path** that routes low-confidence or high-stakes cases to a human.

In [ ]:
TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
print("closed label set -> team:", TEAMS)

## Your Turn
Complete `route`: pick the team from the closed map, and decide when to **escalate** to a human.

In [ ]:
TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}

def route(record):
    intent = record.get("intent", "other")
    team = ___   # TODO: the team for intent, defaulting to "general" (the escape hatch)
    # escalate to a human when the customer is unhappy OR the intent is unknown
    escalate = ___   # TODO: True if sentiment is negative or intent not in TEAMS
    return {"team": team, "escalate": escalate}

try:
    print("refund   ->", route({"intent": "refund", "sentiment": "neutral"}))
    print("delivery ->", route({"intent": "delivery", "sentiment": "negative"}))
    print("unknown  ->", route({"intent": "mystery", "sentiment": "neutral"}))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("a refund routes to billing", lambda: route({"intent": "refund", "sentiment": "neutral"})["team"] == "billing")
expect_true("a delivery routes to logistics", lambda: route({"intent": "delivery", "sentiment": "neutral"})["team"] == "logistics")
expect_true("an unknown intent falls back to general (escape hatch)", lambda: route({"intent": "mystery", "sentiment": "neutral"})["team"] == "general")
expect_true("a negative sentiment escalates to a human", lambda: route({"intent": "delivery", "sentiment": "negative"})["escalate"] is True)
expect_true("a neutral known case does not escalate", lambda: route({"intent": "refund", "sentiment": "neutral"})["escalate"] is False)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Routing makes one agent the front door to a whole system. Routing to the right specialist AGENT is the bridge to Module 8 -- for now it's the label that drives the branch.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>